# Representation study driver

Thin driver for `python -m repstudy.run`. All logic lives in `repstudy/`; every number ends up in
`results/<model>.json` and `results/summary.md`. Needs Linux, one A100/H100 80GB, 16+ vCPUs, ~200 GB disk.
Request gated access first (day 0): `nvidia/Cosmos-Reason2-2B` (also needed by GR00T), `facebook/VGGT-Omega`,
`facebook/dinov3-vitl16-pretrain-lvd1689m`.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path('/content/cs6998-final-proj') if 'google.colab' in sys.modules else Path.cwd()
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/javaapplesauce/cs6998-final-proj', str(REPO)], check=True)
os.chdir(REPO)
os.environ.update(MUJOCO_GL='egl', OMP_NUM_THREADS='1')

In [ ]:
# one venv (see README for why each extra is there)
!pip install -q "lerobot[libero,groot,molmoact2,lingbot_va]==0.6.1" "diffusers==0.39.0" sentencepiece scikit-learn open_clip_torch==3.3.0 git+https://github.com/robbyant/lingbot-vision.git pytest
!pip install -q git+https://github.com/facebookresearch/r3m.git@b2334e726887fa0206962d7984c69c5fb09cceab
!pip install -q --no-deps git+https://github.com/facebookresearch/vggt-omega.git@48b23c8ce72c9a8fcf987f0d7f43d8e4870759f6
from huggingface_hub import login; login()   # gated repos

## Gates G0–G4 (before any production hours)

In [ ]:
def rs(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'repstudy.run', *args], check=check).returncode == 0

!python -m pytest -q tests/test_smoke.py      # G4: CPU end to end, < 60 s
rs('--stage', 'g0')                            # imports, no stdin prompt, assets in place, one EGL frame
rs('--stage', 'g1')                            # dataset contract + decoder parity (downloads lerobot/libero @ a1aaacb7)
rs('--stage', 'g2')                            # first-frame pixel + state parity vs the 50 init states
rs('--stage', 'g3')                            # replayed demo actions succeed at least once

## Blind baseline, G6 pilot, G7

Blind runs twice: the second pass only does work if the §1.5 rule fired (blind > 30% on a suite → that suite
drops proprio for every model, blind included). If G6 fails, edit `CONFIG['head']` and re-run this cell: heads
and evals are keyed by a hash of the head config, so nothing stale is reused.

In [ ]:
for _ in range(2):
    for s in ('train', 'eval'):
        rs('--model', 'blind', '--stage', s)
for s in ('g5', 'calibrate', 'extract', 'train', 'eval'):
    rs('--model', 'vjepa2_1', '--stage', s, '--suites', 'libero_spatial', '--seeds', '0')
rs('--stage', 'g6')                             # pilot clearly above blind, not at floor
rs('--stage', 'g7')                             # measured throughput -> suggested cut level (results/gates.json)

## Production

Order follows §1.9's cut priority: Tier 1, then the §1.5 ablations and resolution check, then the instruction-on
secondary condition, then Tier 2 last (first to be cut; a Tier-2 failure must not stop the run). Token caches are
extracted and consumed one suite at a time (train deletes them), so peak token disk is one suite of one model.
`--cut-level` is persisted in `results/cut_level`; later calls (incl. analyze) reuse it.

In [ ]:
import json
CUT = str(json.loads(Path('results/gates.json').read_text())['G7']['suggested_cut_level'])   # or override
from repstudy.run import CONFIG, apply_cuts, blind_success
cfg = apply_cuts({**CONFIG, 'cut_level': int(CUT)}, blind_success(CONFIG))

def tokens_then_heads(m, text='generic', check=True):
    for su in cfg['suites']:
        for s in ('extract', 'train'):
            rs('--model', m, '--stage', s, '--suites', su, '--text', text, '--cut-level', CUT, check=check)

def run_model(m, check=True):
    if m in cfg['offline_only']:
        return rs('--model', m, '--stage', 'probe', '--cut-level', CUT, check=check)
    for s in ('g5', 'calibrate'):
        rs('--model', m, '--stage', s, '--cut-level', CUT, check=check)
    tokens_then_heads(m, check=check)
    for s in ('eval', 'probe'):
        rs('--model', m, '--stage', s, '--cut-level', CUT, check=check)

for m in cfg['tier1'][1:] + cfg['ablations'] + cfg['offline_only']:     # tier1[0] is blind (done above)
    run_model(m)
for m in cfg['text_models']:                                            # offline instruction-on site curves
    if m in cfg['tier1']:
        rs('--model', m, '--stage', 'calibrate', '--text', 'instruction', '--cut-level', CUT)
if cfg['closed_loop_secondary']:                                        # L3 pair, closed loop
    for m in ('lingbot_va', 'wan22'):
        tokens_then_heads(m, 'instruction')
        rs('--model', m, '--stage', 'eval', '--text', 'instruction', '--cut-level', CUT)
for m in cfg['tier2']:                                                  # last; non-fatal
    run_model(m, check=False)
    if m in cfg['text_models']:
        rs('--model', m, '--stage', 'calibrate', '--text', 'instruction', '--cut-level', CUT, check=False)

## Weight-diff audit, analysis

In [ ]:
rs('--stage', 'weightdiff')     # CPU only; can run any time after downloads
rs('--stage', 'analyze')        # reads the persisted cut level
from IPython.display import Markdown, Image, display
display(Markdown(Path('results/summary.md').read_text()))
for f in sorted(Path('results').glob('fig_*.png')):
    display(Image(str(f)))